# Assignment 28: Q&A RAG Chatbot with Message History
**Student Name:** Patan Khalilulla Khan  
**Course:** GenAI Training Program  

---
## Assignment Overview
This assignment demonstrates building a conversational RAG assistant that maintains context-aware state.


In [ ]:
# Setup: Load, Split, Embed
from langchain_core.documents import Document

# Create sample large docs
texts = ["GenAI is a field of AI that creates new content.", "RAG connects LLMs to external data.", "LangChain provides tools for RAG."]
docs = [Document(page_content=t) for t in texts]

splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=5)
chunks = splitter.split_documents(docs)
vectorstore = FAISS.from_documents(chunks, OpenAIEmbeddings())
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

print("RAG components ready.")


## PART 4 — Q&A RAG Chain with History
Building a chain that accepts user question, retrieves context, and references conversation history.


In [ ]:
# Prompt with History
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only from retrieved context: {context}"),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}")
])

# History store
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# RAG Chain
chain = prompt | llm | StrOutputParser()

# Wrapped with History
rag_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

print("=== Task 9/10: Stateful RAG Test ===")
config = {"configurable": {"session_id": "user1"}}
print("Q1:", rag_with_history.invoke({"context": "GenAI is AI content creation.", "question": "What is GenAI?"}, config=config))
print("Q2 (Follow-up):", rag_with_history.invoke({"context": "GenAI creates content.", "question": "Can you explain more?"}, config=config))


## Task 11: Observations & Insights
1. **Normal RAG vs Conversational RAG**: Normal RAG is stateless (treats queries independently). Conversational RAG uses past context to understand follow-up references.
2. **Role of History**: Enables contextual grounding for ambiguous follow-up questions (e.g., "Tell me more").
3. **Trade-offs**: Long memory improves performance but consumes more tokens and increases latency.
4. **Trimming**: Essential to ensure the history fits within the LLM's context window without losing essential context.
